# Verify Cluster Deployment

End-to-end verification of the custom deep research system running on OpenShift.
Checks backend health, MCP server availability, and frontend route.

## 1. Load Environment

In [ ]:
import os
import subprocess
import json
import httpx
from dotenv import load_dotenv

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path)

NAMESPACE = os.getenv("NAMESPACE", "doc-research-lab")
VERIFY_SSL = os.getenv("VERIFY_SSL", "true").lower() not in ("false", "0", "no")

print(f"Namespace:  {NAMESPACE}")
print(f"Verify SSL: {VERIFY_SSL}")
print("\u2705 Environment loaded")

## 2. Check Pod Health

Verify that all pods in the namespace are in Running state.

In [ ]:
r = subprocess.run(
    ["oc", "get", "pods", "-n", NAMESPACE,
     "-o", "jsonpath={range .items[*]}{.metadata.name}{'\\t'}{.status.phase}{'\\n'}{end}"],
    capture_output=True, text=True,
)

if r.returncode == 0 and r.stdout.strip():
    all_running = True
    for line in r.stdout.strip().split("\n"):
        if line:
            parts = line.split("\t")
            name = parts[0]
            phase = parts[1] if len(parts) > 1 else "Unknown"
            icon = "\u2705" if phase == "Running" else "\u274c"
            print(f"  {icon} {name}: {phase}")
            if phase != "Running":
                all_running = False
    print()
    if all_running:
        print("\u2705 All pods running")
    else:
        print("\u26a0\ufe0f Some pods not ready — check events: oc describe pod <name> -n", NAMESPACE)
else:
    print(f"\u274c No pods found in namespace '{NAMESPACE}'")
    if r.stderr:
        print(r.stderr)

## 3. Verify Backend Health Endpoint

Hit the backend `/health` endpoint to confirm the API is responding.

In [ ]:
# Discover backend route
r = subprocess.run(
    ["oc", "get", "route", "backend", "-n", NAMESPACE,
     "-o", "jsonpath={.spec.host}"],
    capture_output=True, text=True,
)

if r.returncode == 0 and r.stdout.strip():
    backend_url = f"https://{r.stdout.strip()}"
else:
    backend_url = "http://localhost:8000"
    print(f"\u26a0\ufe0f No route found — using {backend_url} (port-forward may be needed)")

print(f"Backend URL: {backend_url}")

try:
    resp = httpx.get(f"{backend_url}/health", timeout=10, verify=VERIFY_SSL)
    if resp.status_code == 200:
        print(f"Response: {resp.json()}")
        print("\u2705 Backend is healthy")
    else:
        print(f"\u274c Backend returned HTTP {resp.status_code}")
except Exception as e:
    print(f"\u274c Cannot reach backend: {e}")
    print(f"   Try: oc port-forward svc/backend 8000:8000 -n {NAMESPACE}")

## 4. Verify MCP Servers (tools/list)

Each MCP server exposes a streamable-http endpoint. Verify they respond to
a `tools/list` request via their internal service URLs.

In [ ]:
mcp_servers = {
    "vector-search-mcp": 9002,
    "web-search-mcp": 9003,
    "verification-mcp": 9004,
    "observability-mcp": 9005,
}

# Use oc exec from the backend pod to reach internal services
r = subprocess.run(
    ["oc", "get", "pods", "-n", NAMESPACE, "-l", "app=backend",
     "-o", "jsonpath={.items[0].metadata.name}"],
    capture_output=True, text=True,
)
backend_pod = r.stdout.strip() if r.returncode == 0 else None

if not backend_pod:
    print("\u26a0\ufe0f No backend pod found — testing via localhost (port-forward needed)")

results = []
for name, port in mcp_servers.items():
    if backend_pod:
        # Reach MCP services via internal DNS from the backend pod
        curl_cmd = (
            f"curl -s -o /dev/null -w '%{{http_code}}' "
            f"http://{name}.{NAMESPACE}.svc:{port}/mcp/ "
            f"-X POST -H 'Content-Type: application/json' "
            f"-d '{{\"jsonrpc\":\"2.0\",\"method\":\"tools/list\",\"id\":1}}'"
        )
        r = subprocess.run(
            ["oc", "exec", backend_pod, "-n", NAMESPACE, "--", "sh", "-c", curl_cmd],
            capture_output=True, text=True,
        )
        status = r.stdout.strip().strip("'")
    else:
        try:
            resp = httpx.post(
                f"http://localhost:{port}/mcp/",
                json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
                timeout=10,
            )
            status = str(resp.status_code)
        except Exception:
            status = "unreachable"

    ok = status == "200"
    icon = "\u2705" if ok else "\u274c"
    print(f"  {icon} {name} (:{port}) — HTTP {status}")
    results.append(ok)

print()
if all(results):
    print("\u2705 All MCP servers responding")
else:
    print("\u26a0\ufe0f Some MCP servers unreachable — check pod logs")

## 5. Verify Frontend Route

Confirm the Chainlit frontend is accessible via its OpenShift route.

In [ ]:
r = subprocess.run(
    ["oc", "get", "route", "frontend", "-n", NAMESPACE,
     "-o", "jsonpath={.spec.host}"],
    capture_output=True, text=True,
)

if r.returncode == 0 and r.stdout.strip():
    frontend_url = f"https://{r.stdout.strip()}"
    print(f"Frontend URL: {frontend_url}")

    try:
        resp = httpx.get(frontend_url, timeout=10, verify=VERIFY_SSL, follow_redirects=True)
        if resp.status_code == 200:
            print("\u2705 Frontend is accessible")
        else:
            print(f"\u26a0\ufe0f Frontend returned HTTP {resp.status_code}")
    except Exception as e:
        print(f"\u274c Cannot reach frontend: {e}")
else:
    print("\u26a0\ufe0f No frontend route found.")
    print(f"   Create one: oc create route edge frontend --service=frontend --port=7860 -n {NAMESPACE}")

## 6. Summary

Print a deployment summary with all verified endpoints.

In [ ]:
print("="*60)
print("  DEPLOYMENT VERIFICATION SUMMARY")
print("="*60)
print(f"  Namespace: {NAMESPACE}")
print(f"  Backend:   {backend_url}")
if r.returncode == 0 and r.stdout.strip():
    print(f"  Frontend:  {frontend_url}")
print(f"  MCP Servers: {sum(results)}/{len(results)} healthy")
print("="*60)

if all(results):
    print("\n\u2705 All services verified — system is operational")
else:
    print("\n\u26a0\ufe0f Some services need attention — review output above")